In [1]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [44]:
pvi_18 = pd.read_csv('data/pres_results/2008-16_pres_2018_dists.csv')
pvi_20 = pd.read_csv('data/pres_results/2008-20_pres_2020_dists.csv')
pvi_24 = pd.read_csv('data/pres_results/2020-24_pres_2024_dists.csv')
pvi_24.head()

,Calculated by The Downballot,Unnamed: 1,Unnamed: 2,Subscribe to our newsletter,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Follow @the-downballot.com on Bluesky,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,District,Incumbent,Party,2024,NaN,NaN,NaN,NaN,NaN,NaN,2020,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Harris,Trump,Total,Harris %,Trump %,Margin,NaN,Biden,Trump,Total,Biden %,Trump %,Margin
2,AK-AL,Nick Begich,(R),"140,026","184,458","338,177",41.41%,54.54%,-13.14%,NaN,"153,778","189,951","357,569",43.01%,53.12%,-10.12%
3,AL-01,Barry Moore,(R),"73,003","257,060","332,700",21.94%,77.26%,-55.32%,NaN,"79,112","243,258","325,715",24.29%,74.68%,-50.40%
4,AL-02,Shomari Figures,(D),"155,603","131,721","290,033",53.65%,45.42%,8.23%,NaN,"174,051","135,333","312,225",55.75%,43.34%,12.40%


In [39]:
pd.set_option('display.max_columns', 100)

In [49]:
# Source: Wikipedia, FEC
# Links:
# https://www.fec.gov/resources/cms-content/documents/federalelections2020.pdf
# https://www.fec.gov/resources/cms-content/documents/federalelections2016.pdf#page=10
# https://www.fec.gov/resources/cms-content/documents/federalelections2012.pdf#page=11
dem_2pv_20 = 81283501 / (81283501 + 74223975) * 100
dem_2pv_16 = 65853514 / (65853514 + 62984828) * 100
dem_2pv_12 = 65915795 / (65915795 + 60933504) * 100
dem_2pv_12, dem_2pv_16, dem_2pv_20

(51.96386225200976, 51.113288930712876, 52.26983492420647)

In [45]:
# 2024 districts wrangling
pvi_24 = pvi_24.iloc[2:]
pvi_24 = pvi_24.set_axis(['district', 'incumbent', 'party', 'dem_24', 'rep_24', 'tot_24', 'dem_pct_24', 'rep_pct_24',
                         'margin_24', 'na1', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20', 'margin_20'], axis=1)
pvi_24 = pvi_24.drop(['na1', 'margin_24', 'margin_20'], axis=1)
for col in ['dem_pct_24', 'rep_pct_24', 'dem_pct_20', 'rep_pct_20']:
    pvi_24[col] = pvi_24[col].str.rstrip('%').astype(float)
for col in ['dem_24', 'rep_24', 'tot_24', 'dem_20', 'rep_20', 'tot_20']:
    pvi_24[col] = pvi_24[col].str.replace(',', '').astype(int)
pvi_24['2p_tot_24'] = pvi_24['dem_24'] + pvi_24['rep_24']
pvi_24['dem_2p_24'] = pvi_24['dem_24'] / pvi_24['2p_tot_24'] * 100
pvi_24['rep_2p_24'] = pvi_24['rep_24'] / pvi_24['2p_tot_24'] * 100
pvi_24['2p_tot_20'] = pvi_24['dem_20'] + pvi_24['rep_20']
pvi_24['dem_2p_20'] = pvi_24['dem_20'] / pvi_24['2p_tot_20'] * 100
pvi_24['rep_2p_20'] = pvi_24['rep_20'] / pvi_24['2p_tot_20'] * 100
pvi_24.head()

,district,incumbent,party,dem_24,rep_24,tot_24,dem_pct_24,rep_pct_24,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,2p_tot_24,dem_2p_24,rep_2p_24,2p_tot_20,dem_2p_20,rep_2p_20
2,AK-AL,Nick Begich,(R),140026,184458,338177,41.41,54.54,153778,189951,357569,43.01,53.12,324484,43.153437,56.846563,343729,44.738151,55.261849
3,AL-01,Barry Moore,(R),73003,257060,332700,21.94,77.26,79112,243258,325715,24.29,74.68,330063,22.117899,77.882101,322370,24.540745,75.459255
4,AL-02,Shomari Figures,(D),155603,131721,290033,53.65,45.42,174051,135333,312225,55.75,43.34,287324,54.155935,45.844065,309384,56.257273,43.742727
5,AL-03,Mike Rogers,(R),82654,229676,314869,26.25,72.94,93357,225360,322031,28.99,69.98,312330,26.463676,73.536324,318717,29.291503,70.708497
6,AL-04,Robert Aderholt,(R),53098,267953,323449,16.42,82.84,60121,262473,325713,18.46,80.58,321051,16.538805,83.461195,322594,18.636738,81.363262


In [47]:
# 2020 districts wrangling
pvi_20 = pvi_20.iloc[1:]
pvi_20 = pvi_20.set_axis(['district', 'incumbent', 'party', 'na0', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20',
                        'na1', 'dem_16', 'rep_16', 'tot_16', 'dem_pct_16', 'rep_pct_16', 'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_20 = pvi_20.drop(['na0', 'na1', 'na2', 'na3'], axis=1)
for yr in ['08', '12']:
    pvi_20 = pvi_20.drop([f'dem_{yr}', f'rep_{yr}', f'tot_{yr}', f'dem_pct_{yr}', f'rep_pct_{yr}'], axis=1)
for col in ['dem_pct_20', 'rep_pct_20', 'dem_pct_16', 'rep_pct_16']:
    pvi_20[col] = pvi_20[col].str.rstrip('%').astype(float)
for col in ['dem_20', 'rep_20', 'tot_20', 'dem_16', 'rep_16', 'tot_16']:
    pvi_20[col] = pvi_20[col].str.replace(',', '').astype(int)
pvi_20['2p_tot_20'] = pvi_20['dem_20'] + pvi_20['rep_20']
pvi_20['dem_2p_20'] = pvi_20['dem_20'] / pvi_20['2p_tot_20'] * 100
pvi_20['rep_2p_20'] = pvi_20['rep_20'] / pvi_20['2p_tot_20'] * 100
pvi_20['2p_tot_16'] = pvi_20['dem_16'] + pvi_20['rep_16']
pvi_20['dem_2p_16'] = pvi_20['dem_16'] / pvi_20['2p_tot_16'] * 100
pvi_20['rep_2p_16'] = pvi_20['rep_16'] / pvi_20['2p_tot_16'] * 100

pvi_20['lean_16'] = pvi_20['dem_2p_16'] - dem_2pv_16
pvi_20['lean_20'] = pvi_20['dem_2p_20'] - dem_2pv_20

pvi_20.head()

,district,incumbent,party,dem_20,rep_20,tot_20,dem_pct_20,rep_pct_20,dem_16,rep_16,tot_16,dem_pct_16,rep_pct_16,2p_tot_20,dem_2p_20,rep_2p_20,2p_tot_16,dem_2p_16,rep_2p_16,lean_16,lean_20
1,AK-AL,Mary Peltola,(R),153778,189951,357569,43.0,53.1,116454,163387,309407,37.6,52.8,343729,44.738151,55.261849,279841,41.614345,58.385655,-9.498944,-7.531684
2,AL-01,Jerry Carl,(R),117136,211370,331886,35.3,63.7,103364,192634,303478,34.1,63.5,328506,35.657187,64.342813,295998,34.920506,65.079494,-16.192783,-16.612648
3,AL-02,Barry Moore,(R),107776,195953,306714,35.1,63.9,94299,185505,285664,33.0,64.9,303729,35.484264,64.515736,279804,33.701806,66.298194,-17.411483,-16.785571
4,AL-03,Mike Rogers,(R),109495,212012,324741,33.7,65.3,93300,188477,288776,32.3,65.3,321507,34.056801,65.943199,281777,33.111290,66.888710,-18.001999,-18.213034
5,AL-04,Robert Aderholt,(R),57133,260535,320725,17.8,81.2,50722,233661,290726,17.4,80.4,317668,17.985129,82.014871,284383,17.835806,82.164194,-33.277483,-34.284706


In [48]:
pvi_18

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,2020,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,2016,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,2012,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,2008,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,District,Incumbent,Party,NaN,Biden,Trump,Total,Biden%,Trump%,NaN,Clinton,Trump,Total,Clinton%,Trump%,NaN,Obama,Romney,Total,Obama%,Romney%,NaN,Obama,McCain,Total,Obama%,McCain%
1,AK-AL,"Young, Don",(R),NaN,"153,778","189,951","357,569",43.0%,53.1%,NaN,"116,454","163,387","309,407",37.6%,52.8%,NaN,"122,640","164,676","297,625",41.2%,55.3%,NaN,"123,594","193,841","324,467",38.1%,59.7%
2,AL-01,"Byrne, Bradley",(R),NaN,"117,136","211,370","331,886",35.3%,63.7%,NaN,"103,364","192,634","303,478",34.1%,63.5%,NaN,"111,735","184,786","298,837",37.4%,61.8%,NaN,"115,975","183,753","301,556",38.5%,60.9%
3,AL-02,"Roby, Martha",(R),NaN,"107,776","195,953","306,714",35.1%,63.9%,NaN,"94,299","185,505","285,664",33.0%,64.9%,NaN,"105,572","182,287","289,864",36.4%,62.9%,NaN,"101,076","186,164","288,807",35.0%,64.5%
4,AL-03,"Rogers, Mike",(R),NaN,"109,495","212,012","324,741",33.7%,65.3%,NaN,"93,300","188,477","288,776",32.3%,65.3%,NaN,"103,089","174,465","280,128",36.8%,62.3%,NaN,"103,221","176,434","281,951",36.6%,62.6%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,WI-08,"Gallagher, Mike",(R),NaN,"174,136","241,140","421,811",41.3%,57.2%,NaN,"142,677","207,620","369,562",38.6%,56.2%,NaN,"177,346","191,127","372,886",47.6%,51.3%,NaN,"194,663","163,343","362,769",53.7%,45.0%
432,WV-01,"McKinley, David",(R),NaN,"81,849","183,904","270,510",30.3%,68.0%,NaN,"64,384","165,934","243,894",26.4%,68.0%,NaN,"81,017","141,736","227,948",35.5%,62.2%,NaN,"102,826","140,421","247,507",41.5%,56.7%
433,WV-02,"Mooney, Alex",(R),NaN,"92,623","184,784","282,556",32.8%,65.4%,NaN,"73,487","164,674","250,267",29.4%,65.8%,NaN,"89,079","140,783","234,596",38.0%,60.0%,NaN,"109,369","136,259","249,130",43.9%,54.7%
434,WV-03,"Miller, Carol",(R),NaN,"61,512","176,694","241,586",25.5%,73.1%,NaN,"50,923","158,763","218,890",23.3%,72.5%,NaN,"68,173","135,136","207,894",32.8%,65.0%,NaN,"91,662","120,786","216,725",42.3%,55.7%


In [ ]:
# 2018 districts wrangling
pvi_18 = pvi_18.iloc[1:]
pvi_18 = pvi_18.set_axis(['district', 'incumbent', 'party', 'na0', 'dem_20', 'rep_20', 'tot_20', 'dem_pct_20', 'rep_pct_20',
                        'na1', 'dem_16', 'rep_16', 'tot_16', 'dem_pct_16', 'rep_pct_16', 'na2', 
                         'dem_12', 'rep_12', 'tot_12', 'dem_pct_12', 'rep_pct_12', 'na3',
                        'dem_08', 'rep_08', 'tot_08', 'dem_pct_08', 'rep_pct_08'], axis=1)
pvi_18 = pvi_18.drop(['na0', 'na1', 'na2', 'na3'], axis=1)
for yr in ['08', '20']:
    pvi_18 = pvi_18.drop([f'dem_{yr}', f'rep_{yr}', f'tot_{yr}', f'dem_pct_{yr}', f'rep_pct_{yr}'], axis=1)
for col in ['dem_pct_12', 'rep_pct_12', 'dem_pct_16', 'rep_pct_16']:
    pvi_18[col] = pvi_18[col].str.rstrip('%').astype(float)
for col in ['dem_12', 'rep_12', 'tot_12', 'dem_16', 'rep_16', 'tot_16']:
    pvi_18[col] = pvi_18[col].str.replace(',', '').astype(int)
pvi_18['2p_tot_12'] = pvi_18['dem_12'] + pvi_18['rep_12']
pvi_18['dem_2p_12'] = pvi_18['dem_12'] / pvi_18['2p_tot_12'] * 100
pvi_18['rep_2p_12'] = pvi_18['rep_12'] / pvi_18['2p_tot_12'] * 100
pvi_18['2p_tot_16'] = pvi_18['dem_16'] + pvi_18['rep_16']
pvi_18['dem_2p_16'] = pvi_18['dem_16'] / pvi_18['2p_tot_16'] * 100
pvi_18['rep_2p_16'] = pvi_18['rep_16'] / pvi_18['2p_tot_16'] * 100

pvi_18['lean_16'] = pvi_18['dem_2p_16'] - dem_2pv_16
pvi_18['lean_12'] = pvi_18['dem_2p_12'] - dem_2pv_12

pvi_18.head()